# FactCG + MiniCheck — full LLM-AggreFact benchmark

This Kaggle notebook integrates the official FactCG DeBERTa-v3-Large checkpoint into MiniCheck's native encoder Inferencer, then evaluates the complete pinned LLM-AggreFact test split (29,320 rows across 11 datasets). The benchmark loop calls only MiniCheck.score().

Protocol: full test split, chunk_size=500, batch_size=8, strict fixed threshold probability > 0.5, primary metric Balanced Accuracy. No threshold or parameter tuning is performed.

The run is designed for a Kaggle T4 with Internet enabled. Accept the LLM-AggreFact access conditions and add HF_TOKEN as a Kaggle Secret. The compact incremental checkpoint supports safe resume without repeatedly writing document text.

In [ ]:
# Reproducible Kaggle setup: install, clone, and pin before importing ML packages.
import os
import subprocess
import sys
from pathlib import Path

MINICHECK_REPO = "https://github.com/Liyan06/MiniCheck.git"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

FACTCG_MODEL_ID = "yaxili96/FactCG-DeBERTa-v3-Large"
FACTCG_MODEL_REVISION = "0430e3509dbd28d2dff7a117c0eae25359ff3e80"
FACTCG_SOURCE_COMMIT = "41f185413312ed9b3904ec03963fe899693498e0"

DATASET_ID = "lytang/LLM-AggreFact"
DATASET_REVISION = "981dfd0"

WORKING_DIR = Path("/kaggle/working")
MINICHECK_DIR = WORKING_DIR / "MiniCheck"
CACHE_DIR = WORKING_DIR / "ckpts"

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ============================================================
# 1. Install dependencies
# ============================================================

packages = [
    "transformers==4.38.1",
    "accelerate==0.27.2",
    "datasets==2.18.0",
    "huggingface_hub==0.21.4",
    "sentencepiece==0.2.0",
    "nltk==3.8.1",
    "scikit-learn==1.4.2",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        *packages,
    ],
    check=True,
)


# ============================================================
# 2. Clone + pin MiniCheck
# ============================================================

if not (MINICHECK_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", MINICHECK_REPO, str(MINICHECK_DIR)],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        str(MINICHECK_DIR),
        "fetch",
        "origin",
        MINICHECK_COMMIT,
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        str(MINICHECK_DIR),
        "checkout",
        "--force",
        "--detach",
        MINICHECK_COMMIT,
    ],
    check=True,
)


# ============================================================
# 3. Install MiniCheck
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        str(MINICHECK_DIR),
    ],
    check=True,
)


# IMPORTANT:
# Make the cloned MiniCheck package visible to THIS Kaggle kernel.
if str(MINICHECK_DIR) not in sys.path:
    sys.path.insert(0, str(MINICHECK_DIR))


# ============================================================
# 4. Verify MiniCheck path
# ============================================================

print(
    "Pinned MiniCheck:",
    subprocess.check_output(
        [
            "git",
            "-C",
            str(MINICHECK_DIR),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip(),
)

print("MiniCheck repo:", MINICHECK_DIR)
print(
    "minicheck package exists:",
    (MINICHECK_DIR / "minicheck").exists(),
)

try:
    import minicheck

    print("MiniCheck import OK")
    print("Imported from:", minicheck.__file__)

except Exception as e:
    print("MiniCheck import FAILED:", repr(e))
    raise


# ============================================================
# 5. Hugging Face token
# ============================================================

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")

    except Exception:
        return None


HF_TOKEN = get_hf_token()

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

    from huggingface_hub import login

    login(
        token=HF_TOKEN,
        add_to_git_credential=False,
    )

    print("HF_TOKEN loaded from Kaggle Secrets.")

else:
    print(
        "HF_TOKEN not found. "
        "The public model can load anonymously; "
        "the gated dataset requires an accepted-access token."
    )

In [ ]:
# Apply a guarded, minimal patch to MiniCheck's existing Inferencer branch.
from pathlib import Path

inference_path = MINICHECK_DIR / "minicheck" / "inference.py"
minicheck_path = MINICHECK_DIR / "minicheck" / "minicheck.py"

def replace_once(text, old, new, description):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f"{description}: expected exactly one source match, found {count}")
    return text.replace(old, new, 1)

text = inference_path.read_text(encoding="utf-8")

text = replace_once(
    text,
    "from typing import List\n",
    """from typing import List

# FactCG contract verified against derenlei/FactCG@41f1854:
# - official HF checkpoint/tokenizer
# - instruction template from factcg/utils.py
# - class 1 is the supported/entailed probability in factcg/inference.py
FACTCG_MODEL_ID = "yaxili96/FactCG-DeBERTa-v3-Large"
FACTCG_MODEL_REVISION = "0430e3509dbd28d2dff7a117c0eae25359ff3e80"
FACTCG_INSTRUCTION_TEMPLATE = (
    '{text_a}\\n\\nChoose your answer: based on the paragraph above can we conclude that '
    '"{text_b}"?\\n\\nOPTIONS:\\n- Yes\\n- No\\nI think the answer is '
)
""",
    "add FactCG constants",
)

text = replace_once(
    text,
    """            elif model_name == 'deberta-v3-large':
                ckpt = 'lytang/MiniCheck-DeBERTa-v3-Large'
                self.max_model_len=2048 if max_model_len is None else max_model_len
                
            else:
""",
    """            elif model_name == 'deberta-v3-large':
                ckpt = 'lytang/MiniCheck-DeBERTa-v3-Large'
                self.max_model_len=2048 if max_model_len is None else max_model_len

            elif model_name == 'FactCG-DeBERTa-v3-Large':
                ckpt = FACTCG_MODEL_ID
                self.max_model_len=2048 if max_model_len is None else max_model_len
                
            else:
""",
    "add FactCG checkpoint branch",
)

text = replace_once(
    text,
    """            config = AutoConfig.from_pretrained(ckpt, num_labels=2, finetuning_task="text-classification", revision='main', token=None, cache_dir=cache_dir)
""",
    """            revision = FACTCG_MODEL_REVISION if model_name == 'FactCG-DeBERTa-v3-Large' else 'main'
            config = AutoConfig.from_pretrained(
                ckpt,
                num_labels=2,
                finetuning_task="text-classification",
                revision=revision,
                token=None,
                cache_dir=cache_dir
            )
""",
    "pin FactCG config revision",
)

if text.count("revision='main'") != 2:
    raise RuntimeError("Expected two remaining tokenizer/model revision='main' occurrences")

text = text.replace("revision='main'", "revision=revision")

# IMPORTANT:
# DeBERTa sequence classification does not support device_map='auto'
encoder_device_map_fragment = (
    ', ignore_mismatched_sizes=False, cache_dir=cache_dir, device_map="auto")'
)
encoder_load_without_device_map = (
    ', ignore_mismatched_sizes=False, cache_dir=cache_dir)'
)
if text.count(encoder_device_map_fragment) != 1:
    raise RuntimeError(
        "Expected exactly one encoder device_map='auto' in pinned inference.py, "
        f"found {text.count(encoder_device_map_fragment)}"
    )
text = text.replace(
    encoder_device_map_fragment,
    encoder_load_without_device_map,
    1
)

text = replace_once(
    text,
    """        self.model.eval()
        self.batch_size = batch_size
""",
    """        if model_name == 'FactCG-DeBERTa-v3-Large':
            if config.num_labels != 2:
                raise ValueError(f"FactCG must expose two labels, got {config.num_labels}")

            config_id2label = {
                int(k): str(v)
                for k, v in config.id2label.items()
            }

            semantic_support_ids = [
                idx
                for idx, label in config_id2label.items()
                if label.lower() in {
                    'supported',
                    'support',
                    'entailment',
                    'entailed',
                    'yes'
                }
            ]

            if semantic_support_ids and semantic_support_ids != [1]:
                raise ValueError(
                    f"FactCG config conflicts with official class-1 support mapping: "
                    f"{config_id2label}"
                )

            self.support_label_id = 1
            self.support_label_source = (
                "derenlei/FactCG@41f1854 factcg/inference.py "
                "softmax(logits)[:, 1]; training orig_label=1 denotes supported"
            )
        else:
            self.support_label_id = 1
            self.support_label_source = "MiniCheck legacy encoder contract"

        self.model.eval()
        self.batch_size = batch_size
""",
    "add source-backed support-label mapping",
)

text = replace_once(
    text,
    """        support_prob_per_chunk = label_probs[:, 1].cpu().numpy()
        max_support_prob = label_probs[:, 1].max().item()
""",
    """        support_prob_per_chunk = label_probs[:, self.support_label_id].cpu().numpy()
        max_support_prob = label_probs[:, self.support_label_id].max().item()
""",
    "route probabilities through verified support class",
)

text = replace_once(
    text,
    """        original_text = [self.tokenizer.eos_token.join([one_doc, one_claim]) for one_doc, one_claim in zip(doc, claim)]

        batch_input = []
""",
    """        if self.model_name == 'FactCG-DeBERTa-v3-Large':
            original_text = [
                FACTCG_INSTRUCTION_TEMPLATE.format(
                    text_a=one_doc,
                    text_b=one_claim
                )
                for one_doc, one_claim in zip(doc, claim)
            ]

            self.tokenizer.pad_token = self.tokenizer.eos_token
        else:
            original_text = [
                self.tokenizer.eos_token.join([one_doc, one_claim])
                for one_doc, one_claim in zip(doc, claim)
            ]

        batch_input = []
""",
    "use official FactCG input formatting",
)

text = replace_once(
    text,
    """            else:
                model_inputs = self.tokenizer(
                    [text for text in mini_batch], 
                    max_length=self.max_model_len, 
                    truncation=True, 
                    padding=True, 
                    return_tensors="pt"
                ) 
""",
    """            elif self.model_name == 'FactCG-DeBERTa-v3-Large':
                model_inputs = self.tokenizer(
                    [text for text in mini_batch],
                    max_length=self.max_model_len,
                    truncation='only_first',
                    padding='longest',
                    return_tensors="pt"
                )
            else:
                model_inputs = self.tokenizer(
                    [text for text in mini_batch], 
                    max_length=self.max_model_len, 
                    truncation=True, 
                    padding=True, 
                    return_tensors="pt"
                ) 
""",
    "use official FactCG tokenizer arguments",
)

text = replace_once(
    text,
    """            batch_org_chunks.append([item[:item.find('</s>')] for item in mini_batch]) 
""",
    """            if self.model_name == 'FactCG-DeBERTa-v3-Large':
                start = len(batch_org_chunks) * self.batch_size
                batch_org_chunks.append(
                    list(doc[start:start + len(mini_batch)])
                )
            else:
                batch_org_chunks.append(
                    [item[:item.find('</s>')] for item in mini_batch]
                )
""",
    "preserve native used_chunks output for FactCG",
)

inference_path.write_text(text, encoding="utf-8")


# ============================================================
# Patch minicheck.py
# ============================================================

text = minicheck_path.read_text(encoding="utf-8")

text = replace_once(
    text,
    "            - 'flan-t5-large'\n            - 'Bespoke-MiniCheck-7B'",
    "            - 'flan-t5-large'\n            - 'FactCG-DeBERTa-v3-Large'\n            - 'Bespoke-MiniCheck-7B'",
    "document FactCG model option",
)

text = replace_once(
    text,
    "                - 'flan-t5-large'\n                    Default: 2048\n                - 'Bespoke-MiniCheck-7B'",
    "                - 'flan-t5-large'\n                    Default: 2048\n                - 'FactCG-DeBERTa-v3-Large'\n                    Default: 2048\n                - 'Bespoke-MiniCheck-7B'",
    "document FactCG maximum length",
)

text = replace_once(
    text,
    "            - 'flan-t5-large'\n                Default: 500\n            - 'Bespoke-MiniCheck-7B'",
    "            - 'flan-t5-large'\n                Default: 500\n            - 'FactCG-DeBERTa-v3-Large'\n                Default: 400\n            - 'Bespoke-MiniCheck-7B'",
    "document FactCG chunk default",
)

old_all = (
    "['roberta-large', 'deberta-v3-large', 'flan-t5-large', "
    "'Bespoke-MiniCheck-7B', 'Granite-Guardian-3.3-8B']"
)

new_all = (
    "['roberta-large', 'deberta-v3-large', 'flan-t5-large', "
    "'FactCG-DeBERTa-v3-Large', "
    "'Bespoke-MiniCheck-7B', 'Granite-Guardian-3.3-8B']"
)

if text.count(old_all) != 2:
    raise RuntimeError(
        f"Expected two complete supported-model lists, "
        f"found {text.count(old_all)}"
    )

text = text.replace(old_all, new_all)

text = replace_once(
    text,
    "if model_name in ['roberta-large', 'deberta-v3-large', 'flan-t5-large']:",
    "if model_name in ['roberta-large', 'deberta-v3-large', 'flan-t5-large', 'FactCG-DeBERTa-v3-Large']:",
    "route FactCG through Inferencer",
)

minicheck_path.write_text(text, encoding="utf-8")


# ============================================================
# Syntax check
# ============================================================

import py_compile

py_compile.compile(
    str(inference_path),
    doraise=True
)

py_compile.compile(
    str(minicheck_path),
    doraise=True
)

print("MiniCheck patched successfully:", inference_path)
print(
    "encoder device_map='auto' removed:",
    encoder_device_map_fragment not in inference_path.read_text()
)

In [ ]:
# Experiment configuration and imports.
import gc
import hashlib
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import nltk
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score

nltk.download("punkt", quiet=True)

from minicheck.minicheck import MiniCheck

MODEL_NAME = "FactCG-DeBERTa-v3-Large"
SPLIT = "test"
EXPECTED_TEST_ROWS = 29320
CHUNK_SIZE = 500
BATCH_SIZE = 8
SCORE_GROUP_SIZE = 25
CHECKPOINT_EVERY = 250

OUTPUT_DIR = Path("/kaggle/working/factcg_minicheck_full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_PATH = OUTPUT_DIR / "predictions_incremental.csv"
FAILURES_PATH = OUTPUT_DIR / "failures.csv"
METRICS_PATH = OUTPUT_DIR / "metrics.csv"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required for the intended Kaggle T4 protocol. Enable GPU and Run All.")
print("GPU:", torch.cuda.get_device_name(0))


## Load the complete pinned test split

source_index is assigned before ordering and is preserved in every output row. No sampling is performed. The compact prediction checkpoint stores source_index, dataset, label, predictions, probabilities, chunk diagnostics, status, and errors; source documents remain in the loaded pinned dataset instead of being rewritten at every checkpoint.

In [ ]:
if not HF_TOKEN:
    raise RuntimeError(
        "LLM-AggreFact is gated. Accept access on Hugging Face, add HF_TOKEN to Kaggle Secrets, and Run All."
    )

raw_test = load_dataset(
    DATASET_ID,
    split=SPLIT,
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)
full_df = raw_test.to_pandas().reset_index(names="source_index")
dataset_names = sorted(full_df["dataset"].unique())

assert len(dataset_names) == 11, (
    f"Expected 11 datasets, found {len(dataset_names)}: {dataset_names}"
)
assert len(full_df) == EXPECTED_TEST_ROWS, (
    f"Pinned test split changed: expected {EXPECTED_TEST_ROWS}, found {len(full_df)}"
)
assert full_df["source_index"].is_unique

# Grouping by dataset makes progress and per-dataset recovery easier while
# source_index retains the original test-split position.
eval_df = (
    full_df.sort_values(["dataset", "source_index"])
    .reset_index(drop=True)
)

selection_sha256 = hashlib.sha256(
    ",".join(map(str, eval_df["source_index"].tolist())).encode("utf-8")
).hexdigest()

distribution = (
    eval_df.groupby(["dataset", "label"])
    .size()
    .unstack(fill_value=0)
)
distribution["Total"] = distribution.sum(axis=1)
display(distribution)
print("Full test rows:", len(eval_df), "| source_index_sha256:", selection_sha256)


## Load FactCG through MiniCheck and prepare compact checkpoint/resume

On rerun, successful rows in /kaggle/working/factcg_minicheck_full/predictions_incremental.csv are skipped. Failed and pending rows are retried. Resume is rejected if its source_index/dataset keys do not exactly match the pinned full test split.

In [ ]:
scorer = MiniCheck(
    model_name=MODEL_NAME,
    batch_size=BATCH_SIZE,
    cache_dir=str(CACHE_DIR),
)
assert scorer.model.model_name == MODEL_NAME
assert scorer.model.support_label_id == 1
print("Checkpoint config id2label:", scorer.model.model.config.id2label)
print("Support-label provenance:", scorer.model.support_label_source)

result_columns = [
    "prediction",
    "support_probability",
    "num_chunks",
    "best_chunk_index",
    "chunk_probabilities",
    "status",
    "error",
    "completed_at_utc",
]
base_columns = ["source_index", "dataset", "label"]

if PREDICTIONS_PATH.exists():
    saved_df = pd.read_csv(PREDICTIONS_PATH)
    required = {*base_columns, *result_columns}
    missing = required.difference(saved_df.columns)
    if missing:
        raise RuntimeError(f"Resume checkpoint is missing columns: {sorted(missing)}")

    expected_keys = set(zip(eval_df["dataset"], eval_df["source_index"].astype(int)))
    saved_keys = set(zip(saved_df["dataset"], saved_df["source_index"].astype(int)))
    if saved_keys != expected_keys:
        raise RuntimeError(
            "Resume checkpoint rows do not match the pinned full test split."
        )

    restored = saved_df.set_index(["dataset", "source_index"])[result_columns]
    results_df = eval_df[base_columns].set_index(["dataset", "source_index"])
    results_df[result_columns] = restored
    results_df = results_df.reset_index()
    print("Resumed successful rows:", results_df["status"].eq("success").sum())
else:
    results_df = eval_df[base_columns].copy()
    results_df["prediction"] = pd.Series(
        [pd.NA] * len(results_df), dtype="Int64"
    )
    results_df["support_probability"] = np.nan
    results_df["num_chunks"] = pd.Series(
        [pd.NA] * len(results_df), dtype="Int64"
    )
    results_df["best_chunk_index"] = pd.Series(
        [pd.NA] * len(results_df), dtype="Int64"
    )
    results_df["chunk_probabilities"] = pd.NA
    results_df["status"] = "pending"
    results_df["error"] = pd.NA
    results_df["completed_at_utc"] = pd.NA

def save_checkpoint():
    tmp_path = PREDICTIONS_PATH.with_suffix(".csv.tmp")
    results_df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, PREDICTIONS_PATH)

    failures = results_df.loc[results_df["status"].eq("failed")].copy()
    failures.to_csv(FAILURES_PATH, index=False)

print(results_df["status"].value_counts(dropna=False))
print("Compact checkpoint columns:", results_df.columns.tolist())


## Full incremental inference using only MiniCheck.score()

Inference uses small 25-row score groups to isolate failures and writes a compact atomic checkpoint every 250 attempted rows. A failed group is retried row-by-row. Failed rows retain missing prediction/probability fields and their real exception—no fallback or fabricated prediction is used.

In [ ]:
def record_success(row_index, pred, prob, chunks, per_chunk):
    flat_probs = np.asarray(per_chunk, dtype=float).reshape(-1)
    if len(chunks) != len(flat_probs):
        raise ValueError(
            f"Output mismatch: {len(chunks)} chunks vs "
            f"{len(flat_probs)} probabilities"
        )

    prob = float(prob)
    pred = int(pred)
    if not np.isfinite(prob) or not 0.0 <= prob <= 1.0:
        raise ValueError(f"Invalid support probability: {prob}")
    if pred != int(prob > 0.5):
        raise ValueError(
            f"Threshold contract violated: pred={pred}, prob={prob}"
        )

    results_df.at[row_index, "prediction"] = pred
    results_df.at[row_index, "support_probability"] = prob
    results_df.at[row_index, "num_chunks"] = len(chunks)
    results_df.at[row_index, "best_chunk_index"] = int(np.argmax(flat_probs))
    results_df.at[row_index, "chunk_probabilities"] = json.dumps(
        flat_probs.tolist()
    )
    results_df.at[row_index, "status"] = "success"
    results_df.at[row_index, "error"] = pd.NA
    results_df.at[row_index, "completed_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

def record_failure(row_index, exc):
    results_df.at[row_index, "prediction"] = pd.NA
    results_df.at[row_index, "support_probability"] = np.nan
    results_df.at[row_index, "num_chunks"] = pd.NA
    results_df.at[row_index, "best_chunk_index"] = pd.NA
    results_df.at[row_index, "chunk_probabilities"] = pd.NA
    results_df.at[row_index, "status"] = "failed"
    results_df.at[row_index, "error"] = (
        f"{type(exc).__name__}: {exc}"[:2000]
    )
    results_df.at[row_index, "completed_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

def score_indices(indices):
    # Text stays in eval_df; the checkpoint remains compact.
    docs = eval_df.loc[indices, "doc"].tolist()
    claims = eval_df.loc[indices, "claim"].tolist()

    preds, probs, chunks, chunk_probs = scorer.score(
        docs=docs,
        claims=claims,
        chunk_size=CHUNK_SIZE,
    )
    if not (
        len(indices)
        == len(preds)
        == len(probs)
        == len(chunks)
        == len(chunk_probs)
    ):
        raise ValueError(
            "MiniCheck.score() returned inconsistent output lengths"
        )

    for row_index, pred, prob, used, per_chunk in zip(
        indices, preds, probs, chunks, chunk_probs
    ):
        record_success(row_index, pred, prob, used, per_chunk)

pending_indices = results_df.index[
    ~results_df["status"].eq("success")
].tolist()
print(f"Rows to process/retry: {len(pending_indices):,}")

# Ensure a valid resume artifact exists even if the run is interrupted early.
save_checkpoint()
attempted_since_checkpoint = 0

for offset in range(0, len(pending_indices), SCORE_GROUP_SIZE):
    group_indices = pending_indices[offset:offset + SCORE_GROUP_SIZE]

    try:
        score_indices(group_indices)
    except Exception as group_exc:
        print(
            f"Group failed ({group_indices[0]}..{group_indices[-1]}): "
            f"{type(group_exc).__name__}: {group_exc}"
        )
        gc.collect()
        torch.cuda.empty_cache()

        for row_index in group_indices:
            try:
                score_indices([row_index])
            except Exception as row_exc:
                record_failure(row_index, row_exc)
                print(
                    "FAILED",
                    results_df.at[row_index, "dataset"],
                    "source_index=",
                    results_df.at[row_index, "source_index"],
                    f"{type(row_exc).__name__}: {row_exc}",
                )
                gc.collect()
                torch.cuda.empty_cache()

    attempted_since_checkpoint += len(group_indices)
    is_last_group = offset + SCORE_GROUP_SIZE >= len(pending_indices)
    if (
        attempted_since_checkpoint >= CHECKPOINT_EVERY
        or is_last_group
    ):
        save_checkpoint()
        attempted_since_checkpoint = 0
        done = int(results_df["status"].eq("success").sum())
        failed = int(results_df["status"].eq("failed").sum())
        print(
            f"Checkpoint: success={done:,}/{len(results_df):,}, "
            f"failed={failed:,}"
        )

save_checkpoint()
print(results_df["status"].value_counts(dropna=False))


## Full-test Balanced Accuracy

Metrics use only genuine successful predictions. Coverage and failure counts are reported beside every dataset score. The macro value is the unweighted mean of the 11 per-dataset Balanced Accuracy scores; if any failures exist it is explicitly marked preliminary.

In [ ]:
metric_rows = []
for dataset_name in dataset_names:
    group = results_df.loc[results_df["dataset"].eq(dataset_name)]
    success = group.loc[group["status"].eq("success")].copy()
    failed_count = int((group["status"] == "failed").sum())
    bacc = np.nan
    metric_error = None
    if len(success) == 0:
        metric_error = "No successful predictions"
    elif success["label"].nunique() < 2:
        metric_error = "Balanced Accuracy undefined: successful rows contain only one class"
    else:
        bacc = balanced_accuracy_score(
            success["label"].astype(int),
            success["prediction"].astype(int),
        ) * 100.0
    metric_rows.append({
        "dataset": dataset_name,
        "n_expected": len(group),
        "n_success": len(success),
        "n_failed": failed_count,
        "coverage": len(success) / len(group),
        "balanced_accuracy_percent": bacc,
        "metric_error": metric_error,
    })

metrics_df = pd.DataFrame(metric_rows)
available = metrics_df["balanced_accuracy_percent"].dropna()
metrics_df.loc[len(metrics_df)] = {
    "dataset": "Macro average",
    "n_expected": int(metrics_df["n_expected"].sum()),
    "n_success": int(metrics_df["n_success"].sum()),
    "n_failed": int(metrics_df["n_failed"].sum()),
    "coverage": metrics_df["n_success"].sum() / metrics_df["n_expected"].sum(),
    "balanced_accuracy_percent": float(available.mean()) if len(available) else np.nan,
    "metric_error": (
        "Preliminary macro over available dataset scores; failures are excluded"
        if metrics_df["n_failed"].sum() > 0 else None
    ),
}
metrics_df.to_csv(METRICS_PATH, index=False)
display(metrics_df)

failure_count = int(results_df["status"].eq("failed").sum())
if failure_count:
    print(f"WARNING: {failure_count} rows failed. See {FAILURES_PATH}; no predictions were fabricated.")
else:
    print(f"All {len(results_df):,} full-test rows completed successfully.")


In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment": "FactCG native MiniCheck / full LLM-AggreFact test split",
    "model_name": MODEL_NAME,
    "model_id": FACTCG_MODEL_ID,
    "model_revision": FACTCG_MODEL_REVISION,
    "factcg_source_repository": "https://github.com/derenlei/FactCG",
    "factcg_source_commit": FACTCG_SOURCE_COMMIT,
    "minicheck_repository": MINICHECK_REPO,
    "minicheck_commit": MINICHECK_COMMIT,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "split": SPLIT,
    "full_test_split": True,
    "expected_test_rows": EXPECTED_TEST_ROWS,
    "test_row_count": len(eval_df),
    "dataset_count": len(dataset_names),
    "source_index_sha256": selection_sha256,
    "batch_size": BATCH_SIZE,
    "score_group_size": SCORE_GROUP_SIZE,
    "checkpoint_every_attempted_rows": CHECKPOINT_EVERY,
    "chunk_size": CHUNK_SIZE,
    "hardware_protocol": "Kaggle T4",
    "primary_metric": "Balanced Accuracy",
    "threshold_rule": (
        "prediction = 1 iff support_probability > 0.5"
    ),
    "threshold_tuned": False,
    "inference_entrypoint": "minicheck.minicheck.MiniCheck.score",
    "support_label_id": scorer.model.support_label_id,
    "support_label_source": scorer.model.support_label_source,
    "checkpoint_id2label": {
        str(k): str(v)
        for k, v in scorer.model.model.config.id2label.items()
    },
    "checkpoint_schema": {
        "documents_saved_in_prediction_csv": False,
        "join_key": ["dataset", "source_index"],
    },
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "nltk": version("nltk"),
    "scikit_learn": version("scikit-learn"),
    "cuda_available": torch.cuda.is_available(),
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "success_count": int(results_df["status"].eq("success").sum()),
    "failure_count": int(results_df["status"].eq("failed").sum()),
    "patched_inference_sha256": sha256_file(inference_path),
    "patched_minicheck_sha256": sha256_file(minicheck_path),
    "outputs": {
        "predictions": str(PREDICTIONS_PATH),
        "failures": str(FAILURES_PATH),
        "metrics": str(METRICS_PATH),
        "manifest": str(MANIFEST_PATH),
    },
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("Saved full-benchmark artifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path)
